In [ ]:
pip install transformers torch scikit-learn sentencepiece

In [3]:
"""Hugging Face Hub에 올라온 사전학습 모델의 설정(config)과 토크나이저를 불러와
이 모델이 어떤 Task(예: 문장 분류)용인지, 입력을 어떻게 처리하는지 확인한다.
모델 가중치는 내려받지 않으므로 가볍게 실행된다.
"""
from transformers import AutoConfig, AutoTokenizer

# Hub 모델 ID: "작성자/모델이름" 형식
# KR-FinBert-SC: 한국어 금융 문장 감성 분류(Sentiment Classification) 모델
name = "snunlp/KR-FinBert-SC"

# AutoConfig: 모델 구조·라벨 정보가 담긴 config.json만 불러온다 (가중치 X)
config = AutoConfig.from_pretrained(name)
# AutoTokenizer: 모델에 맞는 토크나이저 클래스를 자동으로 골라 불러온다
tok = AutoTokenizer.from_pretrained(name)

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--snunlp--KR-FinBert-SC. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an adm

In [5]:
from transformers import pipeline

model = pipeline(
    "text-classification",
    model="snunlp/KR-FinBert-SC"
)

result = model("기업의 영업이익이 크게 증가했다.")
print(result)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 41501.19it/s]


[{'label': 'positive', 'score': 0.9991012811660767}]


In [7]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="joonhyun-kim/roberta-base-klue-ynat-classification",
    device=-1
)

label_map = {
    "LABEL_0": "IT·과학",
    "LABEL_1": "경제",
    "LABEL_2": "사회",
    "LABEL_3": "생활문화",
    "LABEL_4": "세계",
    "LABEL_5": "스포츠",
    "LABEL_6": "정치"
}

texts = [
    "삼성전자 3분기 영업이익 증가",
    "정부, 새로운 경제 정책 발표",
    "프로야구 한국시리즈 우승팀 확정"
]

results = classifier(texts)

for text, result in zip(texts, results):
    label = label_map[result["label"]]

    print("문장:", text)
    print("분류:", label)
    print("확률:", round(result["score"], 4))
    print()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 994.43it/s]


문장: 삼성전자 3분기 영업이익 증가
분류: 경제
확률: 0.9902

문장: 정부, 새로운 경제 정책 발표
분류: 정치
확률: 0.9915

문장: 프로야구 한국시리즈 우승팀 확정
분류: 스포츠
확률: 0.9955



In [11]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

text = "Samsung Electronics reports higher quarterly profit"

labels = [
    "business and financial news",
    "government and political news",
    "sports news",
    "technology and scientific innovation news"
]

result = classifier(
    text,
    candidate_labels=labels,
    hypothesis_template="This article is about {}."
)

for label, score in zip(result["labels"], result["scores"]):
    print(label, round(score, 4))

Loading weights: 100%|██████████| 515/515 [00:01<00:00, 318.71it/s]


business and financial news 0.7784
technology and scientific innovation news 0.1915
government and political news 0.0189
sports news 0.0112


In [14]:
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# -------------------------------------
# 1. 샘플 데이터 (문서 + 메타데이터)
# -------------------------------------

documents = [
    {
        "id": 1,
        "category": "클라우드",
        "title": "DevOps 엔지니어 필수 과목",
        "content": "DevOps 엔지니어는 Linux 운영체제, 네트워크, Docker, Kubernetes, CI/CD, 클라우드 인프라를 학습해야 한다."
    },
    {
        "id": 2,
        "category": "클라우드",
        "title": "시스템 엔지니어 필수 과목",
        "content": "시스템 엔지니어는 운영체제, Linux 서버 관리, TCP/IP 네트워크, 시스템 모니터링, 장애 대응을 학습해야 한다."
    },
    {
        "id": 3,
        "category": "클라우드",
        "title": "클라우드 엔지니어 필수 과목",
        "content": "클라우드 엔지니어는 AWS, Azure, 가상화, 네트워크, IAM, Terraform을 학습해야 한다."
    },
    {
        "id": 4,
        "category": "AI",
        "title": "AI 엔지니어 필수 과목",
        "content": "AI 엔지니어는 Python, 머신러닝, 딥러닝, Transformer, LLM, RAG를 학습해야 한다."
    },
    {
        "id": 5,
        "category": "AI",
        "title": "RAG 엔지니어 필수 과목",
        "content": "RAG 엔지니어는 임베딩, 검색 알고리즘, 벡터 데이터베이스, 프롬프트 엔지니어링을 학습해야 한다."
    },
    {
        "id": 6,
        "category": "데이터분석",
        "title": "데이터 분석가 필수 과목",
        "content": "데이터 분석가는 SQL, Python, 통계학, Pandas, 데이터 시각화를 학습해야 한다."
    },
    {
        "id": 7,
        "category": "데이터분석",
        "title": "데이터 엔지니어 필수 과목",
        "content": "데이터 엔지니어는 SQL, 데이터베이스, ETL, 데이터 파이프라인, Apache Spark를 학습해야 한다."
    },
    {
        "id": 8,
        "category": "보안",
        "title": "보안 엔지니어 필수 과목",
        "content": "보안 엔지니어는 네트워크 보안, 암호학, 접근 제어, 취약점 분석, 로그 분석을 학습해야 한다."
    },
    {
        "id": 9,
        "category": "네트워크",
        "title": "네트워크 엔지니어 필수 과목",
        "content": "네트워크 엔지니어는 TCP/IP, 라우팅, 스위칭, VLAN, DNS, 방화벽을 학습해야 한다."
    }
]

# -------------------------------------
# 2. 제로샷 모델
# -------------------------------------

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=-1
)

labels = ["클라우드", "AI", "데이터분석", "보안", "네트워크"]

# -------------------------------------
# 3. 질문
# -------------------------------------

question = "DevOps 엔지니어에게 가장 필요한 과목은?"

# -------------------------------------
# 4. 제로샷 분류
# -------------------------------------

result = classifier(
    question,
    candidate_labels=labels,
    hypothesis_template="이 질문은 {} 분야에 관한 것이다."
)

category = result["labels"][0]

print("질문:", question)
print("분류 결과:", category)
print("분류 점수:", round(result["scores"][0], 4))

# -------------------------------------
# 5. 메타데이터 필터링
# -------------------------------------

filtered_docs = []

for doc in documents:
    if doc["category"] == category:
        filtered_docs.append(doc)

print("\n필터링된 문서 수:", len(filtered_docs))

for doc in filtered_docs:
    print(doc["title"])

# -------------------------------------
# 6. TF-IDF 검색
# -------------------------------------

if len(filtered_docs) == 0:
    print("해당 분야의 문서가 없습니다.")

else:
    texts = []

    for doc in filtered_docs:
        texts.append(doc["title"] + " " + doc["content"])

    vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 3)
    )

    doc_vectors = vectorizer.fit_transform(texts)
    question_vector = vectorizer.transform([question])

    scores = cosine_similarity(
        question_vector,
        doc_vectors
    )[0]

    best_index = scores.argmax()
    best_doc = filtered_docs[best_index]

    # -------------------------------------
    # 7. 검색 결과 기반 답변
    # -------------------------------------

    print("\n[검색 결과]")
    print("문서:", best_doc["title"])
    print("유사도:", round(float(scores[best_index]), 4))

    print("\n[답변]")
    print(best_doc["content"])

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--MoritzLaurer--mDeBERTa-v3-base-mnli-xnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\t

질문: DevOps 엔지니어에게 가장 필요한 과목은?
분류 결과: 네트워크
분류 점수: 0.4768

필터링된 문서 수: 1
네트워크 엔지니어 필수 과목

[검색 결과]
문서: 네트워크 엔지니어 필수 과목
유사도: 0.399

[답변]
네트워크 엔지니어는 TCP/IP, 라우팅, 스위칭, VLAN, DNS, 방화벽을 학습해야 한다.
